In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "SoccerNet", "--quiet"])

from SoccerNet.Downloader import SoccerNetDownloader
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
SOCCERNET_DIR = REPO_ROOT / "data" / "soccernet"
SOCCERNET_DIR.mkdir(parents=True, exist_ok=True)

downloader = SoccerNetDownloader(LocalDirectory=str(SOCCERNET_DIR))
downloader.password = "TU_CONTRASEÑA_NDA_AQUI"   # <--- tu contraseña

# Descargar el split de test de tracking (49 clips con ground truth)
downloader.downloadDataTask(task="tracking", split=["test"])
print("Descarga completada.")

C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\sgghu\OneDrive\Documents\GitHub\TFG\data\soccernet\tracking\test.zip already exists
Descarga completada.


In [2]:
# ---- Descomprimir el zip de tracking (vino comprimido) ----
import zipfile
from pathlib import Path

SOCCERNET_DIR = REPO_ROOT / "data" / "soccernet"
zip_tracking = SOCCERNET_DIR / "tracking" / "test.zip"

destino = SOCCERNET_DIR / "tracking" / "test"
if not destino.exists():
    print(f"Descomprimiendo {zip_tracking.name} ({zip_tracking.stat().st_size//1024//1024} MB)...")
    with zipfile.ZipFile(zip_tracking, "r") as z:
        z.extractall(SOCCERNET_DIR / "tracking")
    print("Descomprimido.")
else:
    print("Ya estaba descomprimido.")

# Ver la estructura resultante
TEST_DIR = SOCCERNET_DIR / "tracking" / "test"
print(f"\nContenido de tracking/test:")
for p in sorted(TEST_DIR.iterdir())[:10]:
    print(f"  {p.name} {'[DIR]' if p.is_dir() else ''}")

Ya estaba descomprimido.

Contenido de tracking/test:
  SNMOT-116 [DIR]
  SNMOT-117 [DIR]
  SNMOT-118 [DIR]
  SNMOT-119 [DIR]
  SNMOT-120 [DIR]
  SNMOT-121 [DIR]
  SNMOT-122 [DIR]
  SNMOT-123 [DIR]
  SNMOT-124 [DIR]
  SNMOT-125 [DIR]


In [3]:
# ---- Elegir un SEGUNDO clip random (semilla distinta) para validar el test ----
import random, cv2
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
SOCCERNET_DIR = REPO_ROOT / "data" / "soccernet"
TEST_DIR = SOCCERNET_DIR / "tracking" / "test"

clips = sorted([d for d in TEST_DIR.iterdir() if d.is_dir() and d.name.startswith("SNMOT")])

# Excluir el clip ya evaluado (SNMOT-192) para no repetir
YA_EVALUADOS = {"SNMOT-146"}
disponibles = [c for c in clips if c.name not in YA_EVALUADOS]
print(f"Clips disponibles (excluyendo {YA_EVALUADOS}): {len(disponibles)}")

# Semilla FIJA distinta a la de SNMOT-192 (que fue 42) -> reproducible
random.seed(12)
clip = random.choice(disponibles)
print(f"Clip elegido (seed=123): {clip.name}")

# Verificar estructura
img_dir = clip / "img1"
gt_file = clip / "gt" / "gt.txt"
info_file = clip / "gameinfo.ini"
frames_jpg = sorted(img_dir.glob("*.jpg"))
print(f"  Frames: {len(frames_jpg)}")
print(f"  Ground truth: {gt_file.exists()}")
print(f"  gameinfo.ini: {info_file.exists()}")

# Montar el vídeo con OpenCV (sin ffmpeg)
OUT_VIDEO = REPO_ROOT / "data" / "raw" / f"soccernet_{clip.name}.mp4"
primero = cv2.imread(str(frames_jpg[0]))
h, w = primero.shape[:2]
writer = cv2.VideoWriter(str(OUT_VIDEO), cv2.VideoWriter_fourcc(*"mp4v"), 25, (w, h))
for f in frames_jpg:
    writer.write(cv2.imread(str(f)))
writer.release()
print(f"\nVídeo montado: {OUT_VIDEO.name} ({w}x{h}, {len(frames_jpg)} frames)")
print(f"-> Para procesarlo: en fase 7, VIDEO_FILE = 'soccernet_{clip.name}.mp4'")

Clips disponibles (excluyendo {'SNMOT-146'}): 48
Clip elegido (seed=123): SNMOT-147
  Frames: 750
  Ground truth: True
  gameinfo.ini: True

Vídeo montado: soccernet_SNMOT-147.mp4 (1920x1080, 750 frames)
-> Para procesarlo: en fase 7, VIDEO_FILE = 'soccernet_SNMOT-147.mp4'


In [4]:
# ---- CELDA 3: inspeccionar el ground truth y el gameinfo.ini ----
import csv, configparser

# Ground truth: formato MOT (frame, track_id, x, y, w, h, conf, -1, -1, -1)
print("=== GROUND TRUTH (gt.txt) ===")
with open(gt_file) as fh:
    filas = list(csv.reader(fh))
print("Primeras 5 líneas:")
for f in filas[:5]:
    print(f"  {f}")
frames_gt = sorted(set(int(f[0]) for f in filas))
tracks_gt = sorted(set(int(f[1]) for f in filas))
print(f"\nTotal detecciones GT: {len(filas)}")
print(f"Frames anotados: {len(frames_gt)} (de {min(frames_gt)} a {max(frames_gt)})")
print(f"Tracks únicos: {len(tracks_gt)} -> {tracks_gt}")

# gameinfo.ini: roles/equipos de cada track
print("\n=== GAMEINFO.INI (completo) ===")
cfg = configparser.ConfigParser()
cfg.read(info_file)
for sec in cfg.sections():
    print(f"\n[{sec}]")
    for key in cfg[sec]:
        print(f"  {key} = {cfg[sec][key]}")

=== GROUND TRUTH (gt.txt) ===
Primeras 5 líneas:
  ['1', '1', '599', '378', '59', '160', '1', '-1', '-1', '-1']
  ['2', '1', '599', '378', '59', '160', '1', '-1', '-1', '-1']
  ['3', '1', '599', '378', '58', '160', '1', '-1', '-1', '-1']
  ['4', '1', '599', '378', '58', '160', '1', '-1', '-1', '-1']
  ['5', '1', '599', '378', '58', '160', '1', '-1', '-1', '-1']

Total detecciones GT: 6272
Frames anotados: 750 (de 1 a 750)
Tracks únicos: 27 -> [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]

=== GAMEINFO.INI (completo) ===

[Sequence]
  name = SNMOT-147
  gameid = 8
  actionposition = 1360502
  actionclass = Clearance
  visibility = visible
  clipstart = 1354000
  gametimestart = 2 - 22:34
  clipstop = 1384000
  gametimestop = 2 - 23:04
  num_tracklets = 27
  trackletid_1 = goalkeepers team left;za
  trackletid_2 = ball;1
  trackletid_3 = player team right;20
  trackletid_4 = player team left;44
  trackletid_5 = player team right;Z
  t